Imports and dates

In [ ]:
import sys
from pathlib import Path

# Add repo root to PYTHONPATH
HERE = Path().resolve()
PROJECT_ROOT = HERE.parents[0]   # notebooks → repo root
sys.path.insert(0, str(PROJECT_ROOT))


from prm_opt.run_s25 import run_s25_s1, run_s25_s2, run_s25_s2_v2
from prm_opt.run_s26 import run_s26_s1, run_s26_s2
from prm_opt.outputs import build_run_report, print_run_report
from prm_opt.config import PlanningToggles

# -----------------------------
# DATE RANGES
# -----------------------------
START_S25 = "2025-03-30"
END_S25   = "2025-04-02" #"2025-10-26"

START_S26 = "2025-03-29"
END_S26   = "2025-10-24"

Common toggles


In [2]:

toggles = PlanningToggles(
    sla_buffer_mins=0,
    spill_bucket_cap=12,
    standby_dep_vert_mins=10,
    standby_arr_horiz_mins=10,
)


Run S25 Scenario 1 (baseline)

In [3]:
out_s25_s1 = run_s25_s1(START_S25, END_S25, toggles=toggles)

out_s25_s1["summary"]




Unmatched passenger rows after merge (missing Chocks DT): 46
Unique unmatched flight keys: 28

Unmatched key reasons:
reason
no_flight_candidate               24
scheduled_dt_mismatch              3
exact_match_should_have_joined     1
Name: count, dtype: int64

Sample unmatched keys with reasons (top 50):
   Airline Code Flight Number A/D Scheduled Flight DT_prm  \
19           EI          3257   D     2025-03-31 15:30:00   
20           A5          1486   A     2025-03-31 16:35:00   
25           A5          1487   D     2025-04-01 17:25:00   
17           CJ          8710   A     2025-03-30 20:25:00   
15           CJ          8706   A     2025-03-31 17:10:00   
16           CJ          8704   A     2025-04-01 12:45:00   
27           CJ          8714   A     2025-04-01 21:10:00   
2            RR          2885   A     2025-03-30 19:35:00   
18           RR          5161   A     2025-03-31 08:05:00   
6            RR          5160   D     2025-03-31 08:30:00   
24           RR     

{'PeakAmb': 5,
 'PeakMini': 1,
 'PeakDrivers': 5,
 'PeakAmb_bucket': Timestamp('2025-03-30 11:00:00'),
 'PeakMini_bucket': Timestamp('2025-03-30 09:15:00'),
 'PeakDrivers_bucket': Timestamp('2025-03-30 11:00:00'),
 'CurrentAmb': 14,
 'GapAmb': -9,
 'CurrentMini': 3,
 'GapMini': -2}

In [4]:
out_s25_s1["ambulift_curve"]


s
2025-03-30 00:15:00    1
2025-03-30 00:30:00    1
2025-03-30 00:45:00    1
2025-03-30 04:15:00    3
2025-03-30 04:30:00    1
                      ..
2025-04-01 22:30:00    1
2025-04-01 22:45:00    1
2025-04-01 23:00:00    1
2025-04-01 23:15:00    1
2025-04-02 00:00:00    0
Name: _amb, Length: 238, dtype: int64

In [5]:
out_s25_s1["driver_curve"].head()

s
2025-03-30 00:15:00    1
2025-03-30 00:30:00    1
2025-03-30 00:45:00    1
2025-03-30 04:15:00    3
2025-03-30 04:30:00    1
dtype: int64

Run S25 Scenario 2 (optimised)

In [6]:

#out = run_s25_s2_lp(START_S25, END_S25, toggles=toggles, run_lp_ladder=True, solve_milp=False)
#out["lp_baseline"], out["lp_ladder"]


In [ ]:

out = run_s25_s2_v2(
    start=START_S25,
    end=END_S25,
    solver_name="highs",
    toggles=toggles,
    solve_model=True,
    time_limit_sec=780,
    threads=8,
    mip_rel_gap=0.20,  # optional: stop earlier with a usable solution
)

report = build_run_report(out)     # builds sanity + peak day hourly
print_run_report(report)           # prints clean report


# report["peak_day_report"]["hourly"].to_csv("peak_day_hourly_fleet.csv")


PRM OPT — S25 Scenario 2 (v2)
Window : 2025-03-30 → 2025-04-02

[1/6] ingest_s25…

Unmatched passenger rows after merge (missing Chocks DT): 46
Unique unmatched flight keys: 28

Unmatched key reasons:
reason
no_flight_candidate               24
scheduled_dt_mismatch              3
exact_match_should_have_joined     1
Name: count, dtype: int64

Sample unmatched keys with reasons (top 50):
   Airline Code Flight Number A/D Scheduled Flight DT_prm  \
19           EI          3257   D     2025-03-31 15:30:00   
20           A5          1486   A     2025-03-31 16:35:00   
25           A5          1487   D     2025-04-01 17:25:00   
17           CJ          8710   A     2025-03-30 20:25:00   
15           CJ          8706   A     2025-03-31 17:10:00   
16           CJ          8704   A     2025-04-01 12:45:00   
27           CJ          8714   A     2025-04-01 21:10:00   
2            RR          2885   A     2025-03-30 19:35:00   
18           RR          5161   A     2025-03-31 08:05:00  

In [8]:
# out_s25_s2 = run_s25_s2(
#     start=START_S25,
#     end=END_S25,
#     solver_name="highs",
#     toggles=toggles,
#     run_ladder=True, 
#     solve_model=False
# )

# out_s25_s2["summary"]

Inspect raw optimisation decisions

In [16]:
print(vehicle_df)


    vehicle_type   class_id                   flight_key              bucket  \
0            Amb  AMB_14111  AF_1886_2025-03-31 21:15:00 2025-03-31 21:00:00   
1            Amb  AMB_14111   DL_209_2025-03-31 07:15:00 2025-03-31 08:30:00   
2            Amb  AMB_14111    EK_23_2025-03-31 20:00:00 2025-03-31 20:00:00   
3            Amb  AMB_14111  FR_2885_2025-04-01 11:45:00 2025-04-01 11:30:00   
4            Amb  AMB_14111  FR_4525_2025-03-30 18:00:00 2025-03-30 18:00:00   
..           ...        ...                          ...                 ...   
800         Mini   MB_EV_18   U2_342_2025-04-01 17:30:00 2025-04-01 17:15:00   
801         Mini   MB_EV_18   U2_608_2025-03-30 22:00:00 2025-03-30 22:00:00   
802         Mini   MB_EV_18  U2_6307_2025-04-01 21:15:00 2025-04-01 21:15:00   
803         Mini   MB_EV_18    UA_36_2025-03-31 10:00:00 2025-03-31 10:00:00   
804         Mini   MB_EV_18    UA_36_2025-03-31 10:15:00 2025-04-01 00:00:00   

     count  
0        1  
1        1  


In [17]:
print(job_df)

         j  Passenger ID                   flight_key dir    scheduled_bucket  \
0        0      10488504   FR_813_2025-03-31 09:45:00   D 2025-03-31 09:45:00   
1        1      10668811   LS_775_2025-03-30 04:15:00   D 2025-03-30 04:15:00   
2        2      10668812   LS_775_2025-03-30 05:15:00   D 2025-03-30 05:15:00   
3        3      10668820  LS_3921_2025-03-30 05:15:00   D 2025-03-30 05:15:00   
4        4      10668821  LS_3921_2025-03-30 04:45:00   D 2025-03-30 04:45:00   
...    ...           ...                          ...  ..                 ...   
1168  1168      10818728  BA_1452_2025-04-01 18:45:00   A 2025-04-01 18:45:00   
1169  1169      10818954  U2_6307_2025-04-01 21:15:00   A 2025-04-01 21:15:00   
1170  1170      10819097    EK_24_2025-04-01 19:30:00   D 2025-04-01 19:30:00   
1171  1171      10819162  BA_1440_2025-04-01 22:15:00   A 2025-04-01 22:15:00   
1172  1172      10819515  FR_6274_2025-04-01 20:15:00   D 2025-04-01 20:15:00   

          release_bucket   

In [ ]:

# Vertical jobs that used Mini horizontally
out_s25_s2["sanity_checks"]["vertical_with_mini"]



In [ ]:
# SLA breaches by direction
out_s25_s2["sanity_checks"]["sla_breaches_by_dir"]

vehicle allocations

In [ ]:
df_veh_25 = out_s25_s2["vehicle_allocations"]
df_veh_25.head()


Run S26 Scenario 1

In [ ]:

out_s26_s1 = run_s26_s1(
    start=START_S26,
    end=END_S26,
    penetration_rates=penetration_rates,
    ssr_mix=ssr_mix,
    stand_actuals=stand_actuals,
    stand_dist=stand_dist,
    service_time_params=service_time_params,
    chocks_offset_params=chocks_offset_params,
    toggles=toggles,
)

out_s26_s1["summary"]
out_s26_s1["ambulift_curve"].head()
out_s26_s1["driver_curve"].head()


Run S26 scenario 2

In [ ]:

out_s26_s2 = run_s26_s2(
    start=START_S26,
    end=END_S26,
    penetration_rates=penetration_rates,
    ssr_mix=ssr_mix,
    stand_actuals=stand_actuals,
    stand_dist=stand_dist,
    service_time_params=service_time_params,
    chocks_offset_params=chocks_offset_params,
    solver_name="highs",
    toggles=toggles,
)

out_s26_s2["summary"]
